# Main Cloud Orchestrator - Temporal Pseudo-Labeling

This notebook serves as the main orchestrator for running the Fase 0 (Pseudo-Labeling) pipeline on Google Colab or Kaggle. It performs a sparse checkout of the lightweight `experiments/` directory, installs the dependencies, runs the unit tests, and triggers the class-based pseudo-labeling pipeline. All results are synced directly to Google Drive. Once finished (successfully or on error), it automatically disconnects the VM runtime to save credits.

## Cell 1: Shallow Clone and Sparse Checkout
Clones only the lightweight `experiments/` code directory (ignoring heavy dataset or model directories) and installs the package in editable mode.

In [1]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'

# 1. Clone repository sparsely
if not os.path.exists(REPO_NAME):
    print(f"Clonando {REPO_NAME} (solo directorio 'experiments' y sin historial)...")
    !git clone -q --depth 1 --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Actualizando repositorio {REPO_NAME}...")
    %cd {REPO_NAME}
    !git pull -q
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Fallo al navegar al directorio. Ruta actual: {current_dir}")

# 3. Install dependencies in editable mode
print("Instalando el paquete en modo editable con dependencias [cloud]...")
%pip install -q -e .[cloud]

Clonando ia_article (solo directorio 'experiments' y sin historial)...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 4.22 KiB | 4.22 MiB/s, done.
/content/ia_article
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 15 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 4.76 MiB | 12.47 MiB/s, done.
Updating files: 100% (20/20), done.
/content/ia_article/experiments
Instalando el paquete en modo editable con dependencias [cloud]...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 2: Mount Google Drive
Mounts your Google Drive account to access raw CSV/ZIP datasets and API token configs.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3: Run pytest Unit Tests
Executes the colocated unit tests in the repository to guarantee all functions and packages compile properly before starting.

In [3]:
!pytest src/

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/ia_article/experiments
configfile: pyproject.toml
plugins: cov-7.1.0, typeguard-4.5.2, langsmith-0.10.2, anyio-4.14.2
collected 10 items                                                             

src/data_preparation/test_parser.py ..                                   [ 20%]
src/pseudo_labeling/test_pseudo_labeler.py ........                      [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.12.13-final-0 _______________

Name                                         Stmts   Miss Branch BrPart  Cover   Missing
----------------------------------------------------------------------------------------
src/data_preparation/__init__.py                 0      0      0      0   100%
src/data_preparation/parser.py                 176    148   

## Cell 4: Run Pseudo-Labeler Pipeline & Release Resources
Triggers the class-based production pseudo-labeling pipeline. All output clip JSONs are written locally in local VM RAM and uploaded to the Drive checkpoints subfolder (`for_each_clip`) dynamically after each iteration. Resuming is handled natively via Drive queries. Upon completion (or failure), the Google Colab VM is automatically disconnected to save compute credits.

In [4]:
# Runs the class-based pipeline using configured defaults
%run src/pseudo_labeling/pseudo_labeler.py

✓ Google Drive API service initialized.
✓ Found 815 processed clips on Google Drive.
Loading pre-trained YOLO OBB model...
Dynamically mapped vehicle class IDs from yolo26m-obb.pt: [9, 10]
Loading annotations dataset...
✓ split_metadata.csv found. Filtering for 870 'train' split clips (val clips skipped).
Pre-parsing ground truth coordinates for validation check...


100%|██████████| 43389/43389 [00:02<00:00, 16243.17it/s]


Total video clips to process: 870


Processing video clips:  94%|█████████▍| 816/870 [00:27<00:01, 29.94it/s]

[v_xsmkct74d9] OBB Detections: 1981 | Tracks: 183 | Homography failures: 0


...

Processing video clips: 100%|█████████▉| 869/870 [19:32<00:22, 22.04s/it]

[v_zvtywyonpx] OBB Detections: 848 | Tracks: 39 | Homography failures: 0


Processing video clips: 100%|██████████| 870/870 [19:54<00:00,  1.37s/it]

[v_zwkx55dcbi] OBB Detections: 2215 | Tracks: 198 | Homography failures: 0




Merging clip checkpoints into final JSON mapping...
✓ Pipeline completed successfully. Local JSON saved to: /content/tmp_pseudo/static_vehicles.json
Total frames containing static vehicles: 43310
✓ Consolidated JSON uploaded to Google Drive.
Desconectando sesión de Google Colab automáticamente para salvar créditos de cómputo...
